# Acceptance: save any expression, and computing values without `derived_fn`

**This is a sign-off page, not a tutorial.** It checks roadmap steps 40 and 41, done together
on `feat/save-expressions` (plan `plans/save-expressions.plan.md`).

- **Step 40.** `outputs[name] = expr` saves the value of any rate expression that reads time,
  state or flows, during the solve. A seasonal contact rate or a force of infection by age is
  now an output like any other. `SaveFn` and `ComputedValue` are removed.
- **Step 41.** `compile(derived_fn=...)` is removed. To compute several values from the
  parameters, time and state at once, read them through
  `field_refs(defer(f)(Params(), y=StateArray(), t=Time()), Schema)`. `f` runs once per step
  however many rates read it, and parameter-only parts of the model still move to run start.

Nothing here asserts; the checks are in `tests/test_save_expressions.py` and
`tests/test_shared_nodes.py`. Run every cell top to bottom (under a minute). Each section ends
with a **What to check** box.

| | You are checking that… |
|---|---|
| A | a seasonal contact rate saved by expression is the formula at every save time |
| B | a force of infection by age saved by expression equals the old capture route |
| C | in a mixed expression only the part that reads time is saved during the solve |
| D | `field_refs(defer(f)(...))` gives the same model as hand-written rates |
| E | a shared computation is traced once, and parameter products still hoist |

In [ ]:
from typing import Any, NamedTuple

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    FlowModel,
    GroupedOutput,
    OutputSet,
    Param,
    Params,
    Property,
    PropertyMap,
    StateArray,
    Time,
    TransitionFlow,
    defer,
    exp,
    field_refs,
)
from summer4.epi import ForceOfInfection

state = Property("state", ("S", "I", "R"))
age = Property("age", ("child", "adult"))
pmap = PropertyMap.from_property(state).stratify(age)
y0 = np.array([600.0, 380.0, 15.0, 5.0, 0.0, 0.0])
params = {"beta": 0.4, "gamma": 0.1, "amplitude": 0.3}
RUN = dict(t0=0.0, t1=120.0, dt=1.0)

## A. A seasonal contact rate, saved by expression

The contact rate rises and falls over a 60-day season. The same expression object drives the
infection flow and is assigned as an output, so it is saved at every save time. The figure
draws the saved series and the formula, computed in NumPy.

In [ ]:
seasonal = Param("beta") * (1.0 + Param("amplitude") * defer(jnp.cos)(2 * np.pi * Time() / 60.0))
foi = ForceOfInfection("foi", infectious=state["I"], contact_rate=seasonal, group_by=age)

model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], foi))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("gamma")))
cm = model.compile()

outputs = OutputSet()
outputs["contact_rate"] = seasonal
named = cm.run(params, y0, outputs=outputs, **RUN)

days = np.asarray(named["contact_rate"].times.values)
pd.DataFrame(
    {
        "saved: outputs['contact_rate'] = seasonal": np.asarray(named["contact_rate"].array()),
        "formula: 0.4 (1 + 0.3 cos(2 pi t / 60))": 0.4 * (1 + 0.3 * np.cos(2 * np.pi * days / 60)),
    },
    index=pd.Index(days, name="day"),
).plot(title="A. Seasonal contact rate", labels={"value": "contacts per day", "variable": ""}).show()

> **What to check (A).** One line: the saved series lies on the formula, peaking at 0.52 on days
> 0, 60 and 120 and dipping to 0.28 on days 30 and 90.

## B. Force of infection by age, saved by expression

Before step 40 the only way to save a force of infection by age was the capture route: a
`ForceOfInfection` records its value under its own name, saved with `GroupedOutput("foi")`. Now the expression
itself can be an output, and it gives one series per age (`over=age` is read from the
expression). The figure draws both routes for each age.

In [ ]:
by_age = OutputSet()
by_age["saved"] = foi
by_age["captured"] = GroupedOutput("foi")
named_b = cm.run(params, y0, outputs=by_age, **RUN)
print(named_b["saved"].dims)

frame_b = {}
for a in age.traits:
    frame_b[f"{a}: outputs['saved'] = foi"] = np.asarray(
        named_b["saved"].select(age[a]).sum_by().array()
    )
    frame_b[f"{a}: GroupedOutput (capture route)"] = np.asarray(
        named_b["captured"].select(age[a]).sum_by().array()
    )
pd.DataFrame(frame_b, index=pd.Index(days, name="day")).plot(
    title="B. Force of infection by age, two routes",
    labels={"value": "per day", "variable": ""},
).show()

> **What to check (B).** The printed dims are `('time', 'age')`. Each age's two lines are one
> line. The two ages have the same force of infection here (homogeneous mixing), so all four
> lines coincide.

## C. Mixed expressions: only the part that reads time is saved

`Param("k") * Time() * outputs.ref("incidence")` multiplies a saved incidence by something that
depends on time. Python builds `Param("k") * Time()` first, so that whole piece is saved during
the solve, and the multiplication by incidence happens afterwards. A parameter-only factor
(`/ Param("gamma")`) is applied after the solve and saves nothing. The table shows what the
solver is asked to save; the figure compares the result with NumPy on the saved pieces.

In [ ]:
mixed = OutputSet()
mixed.flow("incidence", "infection")
mixed["time_weighted"] = Param("k") * Time() * mixed.ref("incidence")
mixed["per_gamma"] = mixed.ref("incidence") / Param("gamma")
plan = mixed.plan()
display_plan = pd.DataFrame(
    {"saves": [type(r.what).__name__ for r in plan.requests.values()]}, index=list(plan.requests)
)
named_c = cm.run({**params, "k": 0.01}, y0, outputs=mixed, **RUN)
incidence = np.asarray(named_c["incidence"].array())
pd.DataFrame(
    {
        "Param('k') * Time() * ref('incidence')": np.asarray(named_c["time_weighted"].array()),
        "NumPy: 0.01 * t * incidence": 0.01 * days * incidence,
    },
    index=pd.Index(days, name="day"),
).plot(title="C. A time-weighted incidence", labels={"value": "", "variable": ""}).show()
display_plan

> **What to check (C).** The two lines are one line. The table lists two saves: `incidence`
> (`FlowMass`) and `time_weighted` (`SavedExpr`, the `Param('k') * Time()` piece). `per_gamma`
> saves nothing of its own.

## D. Computing several values at once, without `derived_fn`

A function `derive(params, *, y, t)` returns a NamedTuple of values the rates need: here the
seasonal contact rate and a recovery rate that slows as prevalence rises. Before step 41 this
was `compile(derived_fn=derive)` plus `derived_refs(Derived)`. Now the same function, unchanged,
is read through `field_refs(defer(derive)(Params(), y=StateArray(), t=Time()), Derived)`. The
figure compares the infectious count from that model with the same model written out with
ordinary rate expressions.

In [ ]:
class Derived(NamedTuple):
    contact_rate: Any
    recovery_rate: Any


def derive(p: Any, *, y: Any, t: Any) -> Derived:
    contact = p["beta"] * (1.0 + p["amplitude"] * jnp.cos(2 * jnp.pi * t / 60.0))
    prevalence = (y[2] + y[3]) / jnp.sum(y)
    return Derived(contact_rate=contact, recovery_rate=p["gamma"] / (1.0 + prevalence))


D = field_refs(defer(derive)(Params(), y=StateArray(), t=Time()), Derived)


def build(contact, recovery):
    m = FlowModel(pmap)
    m.add_flow(
        TransitionFlow(
            "infection",
            state["S"],
            state["I"],
            ForceOfInfection("foi", infectious=state["I"], contact_rate=contact, group_by=age),
        )
    )
    m.add_flow(TransitionFlow("recovery", state["I"], state["R"], recovery))
    return m.compile()


from summer4 import Reduce

prevalence = Reduce(where=state["I"]) / Reduce()
by_field_refs = build(D.contact_rate, D.recovery_rate)
by_hand = build(seasonal, Param("gamma") / (1.0 + prevalence))
infectious = OutputSet()
infectious.compartments("infectious", state["I"])
pd.DataFrame(
    {
        "field_refs(defer(derive)(...))": np.asarray(
            by_field_refs.run(params, y0, outputs=infectious, **RUN)["infectious"].array()
        ),
        "written as rate expressions": np.asarray(
            by_hand.run(params, y0, outputs=infectious, **RUN)["infectious"].array()
        ),
    },
    index=pd.Index(days, name="day"),
).plot(title="D. Infectious people, two ways to write the model", labels={"value": "people", "variable": ""}).show()

> **What to check (D).** The two lines are one line: the same numbers to the last digit.

## E. Traced once, and parameter work still hoisted

Every rate that reads `D` reads the one `defer(derive)(...)` node, and the vector field
evaluates each node object once per call, so `derive` is traced once (the `cos` count in the
traced program). With `derived_fn` gone, a parameter-only product beside `D` is computed once
per run (it is in the hoist table). The bar chart counts both, for a model where three flows
read `D` and one also multiplies by `Param("beta") * Param("gamma")`.

In [ ]:
three = FlowModel(pmap)
three.add_flow(TransitionFlow("infection", state["S"], state["I"], 0.001 * D.contact_rate))
three.add_flow(TransitionFlow("recovery", state["I"], state["R"], D.recovery_rate))
three.add_flow(
    TransitionFlow(
        "waning", state["R"], state["S"], D.recovery_rate * (Param("beta") * Param("gamma"))
    )
)
cm_e = three.compile()
jaxpr = jax.make_jaxpr(lambda t, y: cm_e.vector_field(t, y, params))(0.0, jnp.asarray(y0))
counts = pd.Series(
    {
        "flows reading D": 3,
        "cos equations traced": sum(e.primitive.name == "cos" for e in jaxpr.jaxpr.eqns),
        "hoist-table entries": len(cm_e.hoist_table.entries),
    }
)
counts.plot.bar(title="E. One trace of derive; the parameter product hoisted", labels={"value": "count", "index": ""}).show()
counts

> **What to check (E).** Three flows read `D`, but there is one `cos` equation. The hoist table
> has one entry: `Param("beta") * Param("gamma")`.